### BLOCK 1 — Install Dependencies

In [52]:
# === Block 1 — Install dependencies (run once if needed) ===
%pip install -U pandas s3fs fsspec fastparquet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\admin\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


### Block 2 — Setup: imports, constants, logging, S3

In [19]:
# === Block 2 — Setup: imports, constants, logging, S3 ===
import os, re, logging, glob
from pathlib import Path

import pandas as pd
import s3fs

# --- TMY3 2024 r2 constants ---
YEAR    = "2024"
DATASET = "resstock_tmy3_release_2"

BASE_TS   = f"s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/{YEAR}/{DATASET}/timeseries_individual_buildings/by_state/upgrade=0"
BASE_META = f"s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/{YEAR}/{DATASET}/metadata_and_annual_results/national/csv"

# --- Output folder ---
OUTDIR = "load_profiles_2024_r2"
os.makedirs(OUTDIR, exist_ok=True)

# --- Logging to console + file ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
log = logging.getLogger("eulp-2024r2")
file_handler = logging.FileHandler(os.path.join(OUTDIR, "errors.log"), encoding="utf-8")
file_handler.setLevel(logging.WARNING)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
log.addHandler(file_handler)

# --- S3 (anonymous access) ---
fs = s3fs.S3FileSystem(anon=True)

print("✓ Block 2: environment is ready")


✓ Block 2: environment is ready


### Block 3 — Loader: read ID list (Excel/CSV)

In [22]:
# === Block 3 — Loader: read ID list (Excel/CSV) ===
# Specify the path to the file with the ID list:
# example for Windows:
ID_PATH = r"C:\Users\admin\Bess_Valuation\id.xlsx"   # ← change if needed
# or if the file is next to the notebook:
# ID_PATH = "id.xlsx"

def load_ids(path_hint: str | Path | None = None) -> pd.DataFrame:
    """
    Reads an Excel/CSV with a list of bldg_id. If there's no 'bldg_id' column — it takes the first column.
    """
    path = Path(path_hint) if path_hint else None
    if not path or not path.exists():
        # auto-search in the current folder
        for pat in ["id.xlsx", "ids.xlsx", "bldg_ids.xlsx", "id.csv", "ids.csv", "bldg_ids.csv"]:
            cand = Path(pat)
            if cand.exists():
                path = cand
                break
        if not path:
            raise FileNotFoundError("ID list file not found. Update the ID_PATH variable.")
    ext = path.suffix.lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    elif ext == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    if "bldg_id" not in df.columns:
        df = df.rename(columns={df.columns[0]: "bldg_id"})
    df["bldg_id"] = df["bldg_id"].astype(int)

    print(f"✓ IDs loaded from: {path} | rows: {len(df)} | first ID: {df['bldg_id'].iloc[0]}")
    return df

ids_df = load_ids(ID_PATH)
first_id = int(ids_df["bldg_id"].iloc[0])



✓ IDs loaded from: C:\Users\admin\Bess_Valuation\id.xlsx | rows: 4435 | first ID: 28


### Block 4 — Loader: read metadata CSV (2024 r2) 

In [27]:
# === Block 4 — Loader: read metadata CSV (2024 r2) ===
def load_meta_2024r2() -> pd.DataFrame | None:
    """
    Returns a metadata DataFrame (if available). We take the first CSV from BASE_META,
    giving priority to those with 'metadata' in the filename.
    """
    try:
        files = fs.ls(BASE_META)
    except FileNotFoundError:
        return None

    csvs = [p for p in files if p.lower().endswith(".csv")]
    csvs = sorted(csvs, key=lambda p: ("metadata" not in p.lower(), p.lower()))
    if not csvs:
        return None

    meta_path = csvs[0]
    try:
        with fs.open(meta_path, "rb") as f:
            dfm = pd.read_csv(f, low_memory=False)
        return dfm
    except Exception as e:
        log.warning(f"Failed to read {meta_path}: {e}")
        return None

meta_df = load_meta_2024r2()
print("✓ Metadata:", "loaded" if meta_df is not None else "unavailable")



✓ Metadata: loaded


### Block 5 — Helpers: list states, parquet path, normalize columns

In [54]:
# === Block 5 — Helpers: states, S3 paths, metadata lookup, column normalization ===
import re
import pandas as pd

def list_states_in_s3() -> list[str]:
    """Return a sorted list of state codes under by_state/upgrade=0."""
    keys = fs.ls(BASE_TS)
    states = []
    for k in keys:
        m = re.search(r"state=([^/]+)/?$", k)
        if m:
            states.append(m.group(1))
    return sorted(set(states))

def parquet_path_for(state_abbr: str, bldg_id: int) -> str:
    """Build the expected Parquet path for a given state and bldg_id in 2024 r2."""
    return f"{BASE_TS}/state={state_abbr}/{int(bldg_id)}-0.parquet"

def find_state_for_id(bldg_id: int, meta_df: pd.DataFrame | None, states_cache: list[str]) -> str | None:
    """
    Determine the state for bldg_id using metadata first; if not found, try by probing S3.
    """
    if meta_df is not None and "bldg_id" in meta_df.columns:
        row = meta_df.loc[meta_df["bldg_id"].astype(int) == int(bldg_id)]
        if not row.empty:
            for cand in ["in.state_abbreviation", "state", "in.state"]:
                if cand in row.columns and pd.notna(row.iloc[0].get(cand)):
                    st = str(row.iloc[0].get(cand)).strip().upper()
                    if len(st) == 2:
                        return st
    for st in states_cache:
        if fs.exists(parquet_path_for(st, bldg_id)):
            return st
    return None

def normalize_timeseries_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Rename key columns to a unified naming and ensure a datetime 'timestamp' exists."""
    if "timestamp" in df.columns:
        tcol = "timestamp"
    elif "time" in df.columns:
        tcol = "time"
    else:
        df["timestamp"] = pd.date_range("2018-01-01", periods=len(df), freq="15min")
        tcol = "timestamp"

    rename = {
        tcol: "timestamp",
        "out.electricity.total.energy_consumption":      "electricity_total_kw",
        "out.electricity.heating.energy_consumption":    "electricity_heating_kw",
        "out.electricity.cooling.energy_consumption":    "electricity_cooling_kw",
        "out.electricity.hot_water.energy_consumption":  "electricity_hot_water_kw",
        "out.electricity.appliances.energy_consumption": "electricity_appliances_kw",
        "out.electricity.lighting.energy_consumption":   "electricity_lighting_kw",
    }
    keep = [k for k in rename if k in df.columns]
    out = df[keep].rename(columns=rename).copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"])
    return out


### Block 6 — Validate profile & download one profile

In [58]:
# === Block 6 — Validate profile & download one profile (fastparquet + s3fs) ===
import os
import pandas as pd

def validate_profile(df: pd.DataFrame) -> list[str]:
    """Return a list of issues (empty list means OK)."""
    issues = []
    if "timestamp" not in df.columns:
        issues.append("Missing column 'timestamp'")
    power_cols = [c for c in df.columns if c.endswith("_kw")]
    if not power_cols:
        issues.append("No consumption columns matching '*_kw'")
    if power_cols and (df[power_cols] < 0).any().any():
        issues.append("Negative values detected in '*_kw' columns")
    if len(df) < int(35040 * 0.95):  # ~ full 15-min TMY year tolerance
        issues.append(f"Incomplete time series: {len(df)} rows (< 35040*0.95)")
    return issues

def _read_parquet_compatible(path: str) -> pd.DataFrame:
    """
    Read Parquet from S3 without relying on pyarrow (uses fastparquet with s3fs open_with).
    """
    try:
        import fastparquet
        pf = fastparquet.ParquetFile(path, open_with=fs.open)  # fs is the s3fs filesystem from Block 2
        return pf.to_pandas()
    except Exception as e_fp:
        # Final fallback: try pandas default (may work if pyarrow is OK in some environments)
        try:
            return pd.read_parquet(path)
        except Exception as e_pd:
            raise ImportError(f"Parquet read failed. fastparquet error: {e_fp}; pandas fallback error: {e_pd}")

def download_one_profile(bldg_id: int, outdir: str = OUTDIR) -> dict:
    """
    Download a TMY3 (15-min) profile for one bldg_id, normalize columns, validate, and save CSV.
    """
    result = {"bldg_id": int(bldg_id), "status": "error", "file_path": None, "state": None, "error": None}
    try:
        states_cache = list_states_in_s3()
        st = find_state_for_id(bldg_id, meta_df, states_cache)
        if not st:
            raise FileNotFoundError(f"Failed to determine state for bldg_id={bldg_id}")

        pq_path = parquet_path_for(st, bldg_id)
        if not fs.exists(pq_path):
            raise FileNotFoundError(f"Parquet not found: {pq_path}")

        df_raw = _read_parquet_compatible(pq_path)
        out = normalize_timeseries_columns(df_raw)

        issues = validate_profile(out)
        if issues:
            log.warning(f"Validation (bldg_id={bldg_id}): {issues}")

        os.makedirs(outdir, exist_ok=True)
        csv_path = os.path.join(outdir, f"bldg_id_{int(bldg_id)}.csv")
        out.to_csv(csv_path, index=False, encoding="utf-8")

        result.update({"state": st, "status": "success", "file_path": csv_path})
        log.info(f"OK → {csv_path} | state={st} | rows={len(out)}")
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
        result["error"] = err
        log.error(f"Failed bldg_id={bldg_id}: {err}")
    return result


### Block 7 — Test: download a single profile

In [60]:
# === Block 7 — Test: download a single profile ===
single_id = 28  # or: single_id = int(ids_df["bldg_id"].iloc[0])
res_single = download_one_profile(single_id)
res_single


2025-11-01 20:57:06,352 - INFO - OK → load_profiles_2024_r2\bldg_id_28.csv | state=MI | rows=35040


{'bldg_id': 28,
 'status': 'success',
 'file_path': 'load_profiles_2024_r2\\bldg_id_28.csv',
 'state': 'MI',
 'error': None}

### Block 8 — Preview: first rows of the saved profile

In [63]:
# === Block 8 — Preview: first rows of the saved profile ===
# Preview first rows of the just-saved profile (for single_id)
test_csv = os.path.join(OUTDIR, f"bldg_id_{int(single_id)}.csv")
if os.path.exists(test_csv):
    df_check = pd.read_csv(test_csv)
    display(df_check.head())
    print("Rows:", len(df_check), "| Columns:", list(df_check.columns))
else:
    print("File not found:", test_csv)



,timestamp,electricity_total_kw,electricity_heating_kw,electricity_cooling_kw,electricity_hot_water_kw
0,2018-01-01 00:15:00,0.115,0.0,0.0,0.0
1,2018-01-01 00:30:00,0.121,0.0,0.0,0.0
2,2018-01-01 00:45:00,0.118,0.0,0.0,0.0
3,2018-01-01 01:00:00,0.138,0.0,0.0,0.0
4,2018-01-01 01:15:00,0.128,0.0,0.0,0.0


Rows: 35040 | Columns: ['timestamp', 'electricity_total_kw', 'electricity_heating_kw', 'electricity_cooling_kw', 'electricity_hot_water_kw']


### Block 9 — IDs: read bldg_id list into ids_df and check dependencies

In [66]:
# === Block 9 — IDs: read bldg_id list into ids_df and check dependencies ===
import os
import pandas as pd
from pathlib import Path

# Specify your path to the ID list file
ID_PATH = Path(r"C:\Users\admin\Bess_Valuation\id.xlsx")   # change if needed

# Folder where profiles are stored (create if not exists)
OUTDIR = "load_profiles_2024_r2"
os.makedirs(OUTDIR, exist_ok=True)

def load_ids_from_file(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    # If there is no 'bldg_id' column — take the first column
    # Clean and cast to int
    ids = (
        pd.read_excel(path) if path.suffix.lower() in (".xlsx", ".xls") else pd.read_csv(path)
    )
    if "bldg_id" not in ids.columns:
        ids = ids.rename(columns={ids.columns[0]: "bldg_id"})
    ids = (
        ids[["bldg_id"]]
        .dropna()
        .astype({"bldg_id": "int64"})
        .drop_duplicates()
        .reset_index(drop=True)
    )
    print(f"✓ ids_df is ready: {len(ids)} rows | first bldg_id = {ids['bldg_id'].iloc[0]}")
    return ids

ids_df = load_ids_from_file(ID_PATH)

# (optional) check that bulk download functions are defined
try:
    download_one_profile, save_status
    print("✓ Functions download_one_profile and save_status are available")
except NameError:
    print("⚠️ Run the blocks where download_one_profile and save_status are defined first")



✓ ids_df is ready: 4435 rows | first bldg_id = 28
⚠️ Run the blocks where download_one_profile and save_status are defined first


### Block 10 — Bulk download of TMY3 profiles

In [69]:
# === Block 10 — Bulk download of TMY3 profiles ===
# Bulk download of 4,435 TMY3 profiles (2024 r2)
# if needed: %pip install -U tqdm

import os, re, time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    from tqdm.auto import tqdm
except Exception:
    # simple no-op if tqdm is not installed
    def tqdm(x, total=None, desc=None): 
        return x

# ---------- SETTINGS ----------
MAX_WORKERS = 8        # parallelism (8–12 is optimal; reduce if bandwidth is weak)
RETRIES     = 3        # retries per ID
SAVE_EVERY  = 100      # how often to append status to CSV
PAUSE_BASE  = 2.0      # base pause between retries (sec), then *2, *4 ...

# ---------- PREPARE LIST ----------
# read all ids (ids_df must be in memory)
all_ids = ids_df["bldg_id"].astype(int).tolist()
print(f"In the ID file: {len(all_ids)} (expect ~4435)")

# already successfully loaded by status
status_path = os.path.join(OUTDIR, "download_status.csv")
skip = set()
if os.path.exists(status_path):
    s = pd.read_csv(status_path)
    ok = s.loc[s["status"]=="success", "bldg_id"].astype(int).tolist()
    skip.update(ok)

# and/or files physically present in the folder
for name in os.listdir(OUTDIR):
    m = re.match(r"bldg_id_(\d+)\.csv$", name)
    if m:
        skip.add(int(m.group(1)))

todo = [bid for bid in all_ids if int(bid) not in skip]
print(f"Already present: {len(skip)} | To download: {len(todo)}")

# ---------- INTERNAL FUNCTIONS ----------
def _download_with_retry(bid: int) -> dict:
    """Downloads one ID with retries."""
    for attempt in range(RETRIES):
        try:
            return download_one_profile(bid, OUTDIR)
        except KeyboardInterrupt:
            raise
        except Exception as e:
            time.sleep(PAUSE_BASE * (2**attempt))  # pause with backoff
            last = e
    return {"bldg_id": bid, "status": "error", "error": str(last)}

def save_status(rows: list[dict], path: str = None):
    path = path or os.path.join(OUTDIR, "download_status.csv")
    df = pd.DataFrame(rows)
    mode = "a" if os.path.exists(path) else "w"
    header = not os.path.exists(path)
    df.to_csv(path, mode=mode, header=header, index=False, encoding="utf-8")
    print(f"✓ Status saved: {path}")

# ---------- RUN ----------
results_buffer = []

def _flush():
    if results_buffer:
        save_status(results_buffer)
        results_buffer.clear()

if not todo:
    print("No new IDs — everything is already downloaded.")
else:
    try:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(_download_with_retry, bid) for bid in todo]
            for i, fut in enumerate(tqdm(as_completed(futures), total=len(futures))):
                r = fut.result()
                results_buffer.append(r)
                if (i+1) % SAVE_EVERY == 0:
                    _flush()
        _flush()
    except KeyboardInterrupt:
        print("⛔ Stopped by user. Saving intermediate status…")
        _flush()

# ---------- SUMMARY ----------
if os.path.exists(status_path):
    df_status = pd.read_csv(status_path)
    total_ok  = (df_status["status"]=="success").sum()
    total_bad = (df_status["status"]!="success").sum()
    print("\n=== SUMMARY ===")
    print(df_status["status"].value_counts(dropna=False))
    print(f"Succeeded: {total_ok} | Errors: {total_bad}")
    # show the first few errors
    bad = df_status[df_status["status"]!="success"].head(10)
    if len(bad):
        print("\nFirst errors:")
        display(bad)



In the ID file: 4435 (expect ~4435)
Already present: 4435 | To download: 0
No new IDs — everything is already downloaded.

=== SUMMARY ===
status
success    3983
Name: count, dtype: int64
Succeeded: 3983 | Errors: 0


### Block 11 — Backfill missing TMY3 profiles

In [72]:
# === Block 11 — Backfill missing TMY3 profiles ===
# BACKFILL MISSING TMY3 PROFILES (2024 r2)
# If needed: %pip install s3fs pyarrow pandas

import os, re, io, time, glob
import pandas as pd
import s3fs
from pathlib import Path

# --------- CONFIG ---------
YEAR    = 2024
DATASET = "resstock_tmy3_release_2"
OUTDIR  = r"C:\Users\admin\load_profiles_2024_r2"  # folder with profiles
ID_PATH = r"C:\Users\admin\Bess_Valuation\id.xlsx" # file with the list of bldg_id (if ids_df is not in memory)

os.makedirs(OUTDIR, exist_ok=True)

# S3 locations
BASE_TS   = f"oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/{YEAR}/{DATASET}/timeseries_individual_buildings/by_state/upgrade=0"
META_PQ   = f"s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/{YEAR}/{DATASET}/metadata_and_annual_results/national/parquet/national_metadata.parquet"
fs = s3fs.S3FileSystem(anon=True)

def parquet_path_for(state: str, bldg_id: int) -> str:
    # In 2024 r2 inside a state folder, files typically look like "<bldg_id>-0.parquet"
    return f"s3://{BASE_TS}/state={state}/{bldg_id}-0.parquet"

def load_meta_df() -> pd.DataFrame | None:
    """Attempt to read national metadata (Parquet). Returns only the required columns."""
    try:
        df = pd.read_parquet(META_PQ, filesystem=fs, columns=["bldg_id","in.state_abbreviation","in.state","state"])
        return df
    except Exception as e:
        print(f"⚠️ Failed to read Parquet metadata: {e}")
        return None

def find_state_for_id(bid: int, meta: pd.DataFrame | None, states_cache: list[str]) -> str | None:
    """Determine the state for bldg_id: try metadata first, then iterate over S3"""
    if meta is not None:
        row = meta.loc[meta["bldg_id"].astype(int) == int(bid)]
        if not row.empty:
            for cand in ["in.state_abbreviation","state","in.state"]:
                if cand in row.columns:
                    val = row.iloc[0].get(cand)
                    if pd.notna(val):
                        st = str(val).strip().upper()
                        if len(st) == 2:
                            return st
    # fallback: iterate over states in S3
    for st in states_cache:
        if fs.exists(parquet_path_for(st, bid)):
            return st
    return None

def normalize_timeseries_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Rename key columns to a unified naming and add timestamp."""
    # detect time column or create synthetic index
    if "timestamp" in df.columns:
        tcol = "timestamp"
    elif "time" in df.columns:
        tcol = "time"
    else:
        # otherwise — create a 15-minute sequence starting from 2018-01-01
        df["timestamp"] = pd.date_range("2018-01-01", periods=len(df), freq="15min")
        tcol = "timestamp"

    rename = {
        tcol: "timestamp",
        "out.electricity.total.energy_consumption":   "electricity_total_kw",
        "out.electricity.heating.energy_consumption": "electricity_heating_kw",
        "out.electricity.cooling.energy_consumption": "electricity_cooling_kw",
        "out.electricity.hot_water.energy_consumption": "electricity_hot_water_kw",
        "out.electricity.appliances.energy_consumption": "electricity_appliances_kw",
        "out.electricity.lighting.energy_consumption":   "electricity_lighting_kw",
    }
    keep = [k for k in rename if k in df.columns]
    out = df[keep].rename(columns=rename).copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"])
    return out

def download_one_profile_2024r2(bid: int, meta_df: pd.DataFrame | None, states_cache: list[str]) -> dict:
    """Download one bldg_id → CSV into OUTDIR."""
    res = {"bldg_id": int(bid), "status": "error", "file_path": None, "state": None, "error": None}
    try:
        st = find_state_for_id(bid, meta_df, states_cache)
        if not st:
            raise FileNotFoundError(f"Failed to determine state for bldg_id={bid}")
        pq = parquet_path_for(st, bid)
        if not fs.exists(pq):
            raise FileNotFoundError(f"Parquet not found: {pq}")

        with fs.open(pq, "rb") as f:
            df = pd.read_parquet(f)

        out = normalize_timeseries_columns(df)
        csv_path = os.path.join(OUTDIR, f"bldg_id_{int(bid)}.csv")
        out.to_csv(csv_path, index=False, encoding="utf-8")

        res.update({"status": "success", "file_path": csv_path, "state": st})
    except Exception as e:
        res["error"] = f"{type(e).__name__}: {e}"
    return res

def save_status(rows: list[dict], path: str = None):
    """Append/create backfill status log."""
    path = path or os.path.join(OUTDIR, "download_status.csv")
    df = pd.DataFrame(rows)
    mode = "a" if os.path.exists(path) else "w"
    header = not os.path.exists(path)
    df.to_csv(path, mode=mode, header=header, index=False, encoding="utf-8")
    print(f"✓ Status saved: {path}")

# --------- 1) Determine what is missing ---------
expected = (
    pd.read_excel(ID_PATH)[["bldg_id"]]
    .dropna()
    .astype({"bldg_id":"int64"})
    .drop_duplicates()
    .squeeze()
    .tolist()
)
present = set()
for name in os.listdir(OUTDIR):
    m = re.match(r"bldg_id_(\d+)\.csv$", name)
    if m:
        present.add(int(m.group(1)))
missing = [bid for bid in expected if int(bid) not in present]
print(f"Expected: {len(expected)} | Present: {len(present)} | Missing: {len(missing)}")

if not missing:
    print("✅ All expected files are present; nothing to backfill.")
else:
    # --------- 2) Prepare lookups ---------
    states_cache = list_states_in_s3()
    meta_df      = load_meta_df()  # can be None — then fallback to S3

    # --------- 3) Download missing ---------
    results = []
    for i, bid in enumerate(missing, 1):
        r = download_one_profile_2024r2(int(bid), meta_df, states_cache)
        results.append(r)
        if i % 50 == 0 or i == len(missing):
            print(f"Downloaded {i}/{len(missing)} | success: {sum(x['status']=='success' for x in results)}")
            # intermediate status save
            save_status(results)
            results.clear()

    # --------- 4) Final status ---------
    if results:
        save_status(results)

    # Summary
    st = pd.read_csv(os.path.join(OUTDIR, "download_status.csv"))
    ok  = (st["status"] == "success").sum()
    bad = (st["status"] != "success").sum()
    print(f"\nSUMMARY: success={ok} | failed={bad}")
    if bad:
        display(st[st["status"]!="success"].head(10))



Expected: 4435 | Present: 4434 | Missing: 1
⚠️ Failed to read Parquet metadata: filesystem is not implemented for the fastparquet engine.
Downloaded 1/1 | success: 1
✓ Status saved: C:\Users\admin\load_profiles_2024_r2\download_status.csv

SUMMARY: success=3983 | failed=1


,bldg_id,state,status,file_path,error
3983,28,success,C:\Users\admin\load_profiles_2024_r2\bldg_id_2...,MI,NaN


### Block 12 — Build hourly sum (kW) matrix by hour

In [75]:
# === Block 12 — Build hourly sum (kW) matrix by hour ===
import os, glob
import pandas as pd
import numpy as np

# === SETTINGS ===
SRC_DIR   = r"C:\Users\admin\load_profiles_2024_r2"   # your folder with profiles
PATTERN   = "bldg_id_*.csv"
OUT_CSV   = os.path.join(SRC_DIR, "hourly_sum_kw_matrix.csv")

# === FILE COLLECTION ===
files = sorted(glob.glob(os.path.join(SRC_DIR, PATTERN)))
if not files:
    raise FileNotFoundError(f"No files found matching pattern {PATTERN} in {SRC_DIR}")
print(f"Files found: {len(files)}")

# === BASE HOURLY INDEX (from the first file) ===
df0 = pd.read_csv(files[0], usecols=["timestamp", "electricity_total_kw"])
df0["timestamp"] = pd.to_datetime(df0["timestamp"])
idx = pd.date_range(
    start=df0["timestamp"].min().floor("H"),
    end=df0["timestamp"].max().ceil("H"),
    freq="H",
    inclusive="left"
)
hourly_sum = pd.DataFrame(index=idx)

# === PROCESS EACH FILE ===
for i, path in enumerate(files, 1):
    df = pd.read_csv(path, usecols=["timestamp", "electricity_total_kw"])
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index()

    # SUM per hour (kW values)
    s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
    col_name = os.path.splitext(os.path.basename(path))[0]  # e.g., "bldg_id_28"
    s_hour.name = col_name

    hourly_sum = hourly_sum.join(s_hour, how="left")

    if i % 200 == 0 or i == len(files):
        print(f"Processed: {i}/{len(files)}")

# === WRITE OUT ===
hourly_sum = hourly_sum.reset_index().rename(columns={"index": "timestamp"})
hourly_sum.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"✓ Saved: {OUT_CSV} | shape = {hourly_sum.shape}")



Files found: 4435


C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  start=df0["timestamp"].min().floor("H"),
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:22: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  end=df0["timestamp"].max().ceil("H"),
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:20: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  idx = pd.date_range(
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use '

Processed: 200/4435


C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Tem

Processed: 400/4435


...

Processed: 4200/4435


C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Tem

Processed: 4400/4435


C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Temp\ipykernel_37844\4021556611.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s_hour = df["electricity_total_kw"].resample("H").sum().astype("float32")
C:\Users\admin\AppData\Local\Tem

Processed: 4435/4435
✓ Saved: C:\Users\admin\load_profiles_2024_r2\hourly_sum_kw_matrix.csv | shape = (8760, 4436)
